In [0]:
from pyspark.sql.types import StructField, StructType, DecimalType,IntegerType,StringType
from pyspark.sql import functions as F


catalogue_name = 'ecommerce'

In [0]:
slvr_brands = spark.table(f"{catalogue_name}.silver.slvr_brands")
slvr_customer = spark.table(f"{catalogue_name}.silver.slvr_customer")
slvr_product = spark.table(f"{catalogue_name}.silver.slvr_product")
slvr_category= spark.table(f"{catalogue_name}.silver.slvr_category")

In [0]:
slvr_brands.createOrReplaceTempView("v_brands")
slvr_customer.createOrReplaceTempView("v_customer")
slvr_product.createOrReplaceTempView("v_product")
slvr_category.createOrReplaceTempView("v_category")

In [0]:
spark.sql(f"use catalog {catalogue_name}")
spark.sql(f"use schema gold")

DataFrame[]

In [0]:
%sql 
create or replace table gold.gld_dim_products as 

with brands_category as(
    select b.brand_name,
        b.brand_code,
        c.category_code,
        c.category_name
    from v_brands b
    inner join v_category c
    on b.category_code = c.category_code
)
select p.* ,
coalesce(bc.brand_name,'Not Available') as brand_name,
coalesce(bc.category_name, 'Not Available') as category_name 
from v_product p
left join brands_category bc
on p.brand_code = bc.brand_code

num_affected_rows,num_inserted_rows


In [0]:
# India States
indian_region ={
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}

# Australia States
australian_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# u ited Kingdom States
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}


# combining into master dictionary

country_state_map = {
    'India': indian_region,
    'Australia': australian_region,
    'United Kingdom': uk_region,
    'United States': us_region,
    'UAE': uae_region,
    'Singapore': singapore_region,
    'Canada': canada_region
}

In [0]:
print(country_state_map)

{'India': {'MH': 'West', 'GJ': 'West', 'RJ': 'West', 'KA': 'South', 'TN': 'South', 'TS': 'South', 'AP': 'South', 'KL': 'South', 'UP': 'North', 'WB': 'North', 'DL': 'North'}, 'Australia': {'VIC': 'SouthEast', 'WA': 'West', 'NSW': 'East', 'QLD': 'NorthEast'}, 'United Kingdom': {'ENG': 'England', 'WLS': 'Wales', 'NIR': 'Northern Ireland', 'SCT': 'Scotland'}, 'United States': {'MA': 'NorthEast', 'FL': 'South', 'NJ': 'NorthEast', 'CA': 'West', 'NY': 'NorthEast', 'TX': 'South'}, 'UAE': {'AUH': 'Abu Dhabi', 'DU': 'Dubai', 'SHJ': 'Sharjah'}, 'Singapore': {'SG': 'Singapore'}, 'Canada': {'BC': 'West', 'AB': 'West', 'ON': 'East', 'QC': 'East', 'NS': 'East', 'IL': 'Other'}}


In [0]:
from pyspark.sql import Row
# 1 Flatten country_state_map into a list of Rows
rows =[]
for country,state in country_state_map.items():
    # print(f"country: {country}, State: {state}")
    for state_code,region in state.items():
        # print(f"country: {country}, State: {state_code}, region: {region}")
        rows.append(Row(country=country,state=state_code,region=region))

print(rows)

[Row(country='India', state='MH', region='West'), Row(country='India', state='GJ', region='West'), Row(country='India', state='RJ', region='West'), Row(country='India', state='KA', region='South'), Row(country='India', state='TN', region='South'), Row(country='India', state='TS', region='South'), Row(country='India', state='AP', region='South'), Row(country='India', state='KL', region='South'), Row(country='India', state='UP', region='North'), Row(country='India', state='WB', region='North'), Row(country='India', state='DL', region='North'), Row(country='Australia', state='VIC', region='SouthEast'), Row(country='Australia', state='WA', region='West'), Row(country='Australia', state='NSW', region='East'), Row(country='Australia', state='QLD', region='NorthEast'), Row(country='United Kingdom', state='ENG', region='England'), Row(country='United Kingdom', state='WLS', region='Wales'), Row(country='United Kingdom', state='NIR', region='Northern Ireland'), Row(country='United Kingdom', stat

In [0]:
# 2. Create mapping dataframe using the flatten list
map_df = spark.createDataFrame(rows)

display(map_df.limit(10))

country,state,region
India,MH,West
India,GJ,West
India,RJ,West
India,KA,South
India,TN,South
India,TS,South
India,AP,South
India,KL,South
India,UP,North
India,WB,North


In [0]:
df_gold = slvr_customer.join(map_df, on =['country','state'],how = 'left')
df_gold = df_gold.fillna({'region': 'Other'})
display(df_gold.limit(5))

country,state,id,phone,country_code,_source_file,_ingested_at,region
India,MH,CUST000000000001,917280033536.0,IN,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-08-05T07:50:15.151Z,West
Australia,VIC,CUST000000000002,619489725433.0,AU,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-08-05T07:50:15.151Z,SouthEast
India,TN,CUST000000000003,919390066524.0,IN,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-08-05T07:50:15.151Z,South
India,TN,CUST000000000004,917073741793.0,IN,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-08-05T07:50:15.151Z,South
Australia,WA,CUST000000000005,618478772532.0,AU,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-08-05T07:50:15.151Z,West


In [0]:
df_gold.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema','true')\
    .saveAsTable(f'{catalogue_name}.gold.gld_dim_customers')

In [0]:
slvr_date = spark.table(f"{catalogue_name}.silver.slvr_date")
display(slvr_date)

date,year,day_name,quarter,week,_source_file,_ingested_at
2025-08-04,2025,Monday,Q3-2025,Week32-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z
2025-08-07,2025,Thursday,Q3-2025,Week32-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z
2025-10-13,2025,Monday,Q4-2025,Week42-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z
2025-10-20,2025,Monday,Q4-2025,Week43-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z
2025-08-03,2025,Sunday,Q3-2025,Week31-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z
2025-09-03,2025,Wednesday,Q3-2025,Week36-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z
2025-09-12,2025,Friday,Q3-2025,Week37-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z
2025-09-16,2025,Tuesday,Q3-2025,Week38-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z
2025-09-30,2025,Tuesday,Q3-2025,Week40-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z
2025-10-12,2025,Sunday,Q4-2025,Week41-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z


In [0]:
gld_date = slvr_date.withColumn('is_weekend',F.when(F.col('day_name').isin('Saturday','Sunday'), F.lit(1))\
                                .otherwise(F.lit(0)))\
                    .withColumn("date_id", F.date_format(F.col("date"), "yyyyMMdd"))\
                    .withColumn("month_name", F.date_format(F.col("date"), "MMMM"))
display(gld_date.limit(5))

date,year,day_name,quarter,week,_source_file,_ingested_at,is_weekend,date_id,month_name
2025-08-04,2025,Monday,Q3-2025,Week32-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z,0,20250804,August
2025-08-07,2025,Thursday,Q3-2025,Week32-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z,0,20250807,August
2025-10-13,2025,Monday,Q4-2025,Week42-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z,0,20251013,October
2025-10-20,2025,Monday,Q4-2025,Week43-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z,0,20251020,October
2025-08-03,2025,Sunday,Q3-2025,Week31-2025,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:06:01.080Z,1,20250803,August


In [0]:
desired_columns_order = [
    "date_id", "date", "year", "month_name", "day_name", "is_weekend", "quarter", "week", "_ingested_at", "_source_file"
]

gld_date  = gld_date.select(desired_columns_order)

In [0]:
gld_date.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalogue_name}.gold.gld_dim_date")

In [0]:
# had a table with wrong name so dropped it

# %sql
# drop table ecommerce.gold.gld_customers;